# 🛡️ ExamGuard AI: YOLOv11 Anti-Cheating Model Training Pipeline

This notebook trains a custom **YOLOv11 Nano** model on Google Colab (Free T4 GPU) and exports it directly to **TensorFlow.js** (`model.json` + `.bin`) for deployment in **ExamGuard AI**.

### Classes Detected:
1. `candidate_face` (Student's real face)
2. `second_person` (Accomplice/helper in the room)
3. `cell_phone` (Smartphones/tablets held in hand)
4. `earphone_headset` (AirPods, in-ear Bluetooth buds)
5. `book_notes` (Cheat sheets, physical notes)
6. `background_photo` (Wall posters, framed portraits — suppresses false alarms)

## Step 1: Check GPU & Install Required Libraries

In [ ]:
# Verify GPU is enabled (should show NVIDIA Tesla T4)
!nvidia-smi

# Install Ultralytics YOLOv11, TensorFlow.js converter, and Roboflow
!pip install -q ultralytics roboflow tensorflowjs onnx

## Step 2: Download the Anti-Cheating Dataset
You can either use a free Roboflow dataset or upload your own images.

In [ ]:
import os

# Option A: Download from Roboflow Universe (Recommended)
# Replace with your dataset URL or API key from universe.roboflow.com
from roboflow import Roboflow

# If you have your Roboflow API Key, paste it here:
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

if ROBOFLOW_API_KEY != "YOUR_ROBOFLOW_API_KEY":
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("anti-cheating-ai").project("exam-proctoring-detection")
    dataset = project.version(1).download("yolov11")
    data_yaml_path = f"{dataset.location}/data.yaml"
else:
    print("[*] No Roboflow key provided. Generating sample dataset template...")
    # Generates standard local YAML configuration
    os.makedirs("/content/dataset/images/train", exist_ok=True)
    os.makedirs("/content/dataset/images/val", exist_ok=True)
    os.makedirs("/content/dataset/labels/train", exist_ok=True)
    os.makedirs("/content/dataset/labels/val", exist_ok=True)
    
    data_yaml_path = "/content/dataset/data.yaml"
    with open(data_yaml_path, "w") as f:
        f.write("""path: /content/dataset
train: images/train
val: images/val
names:
  0: candidate_face
  1: second_person
  2: cell_phone
  3: earphone_headset
  4: book_notes
  5: background_photo
""")
    print(f"[✓] Template YAML ready at: {data_yaml_path}")

## Step 3: Train YOLOv11 Nano with GPU Acceleration

In [ ]:
from ultralytics import YOLO

# 1. Load the pre-trained YOLOv11 Nano weights
model = YOLO('yolo11n.pt')

# 2. Launch transfer learning
results = model.train(
    data=data_yaml_path,
    epochs=50,             # 50 epochs is standard
    imgsz=640,             # 640x640 webcam resolution
    batch=16,              # Adjust if needed
    device=0,              # Uses NVIDIA GPU
    patience=10,           # Early stopping
    augment=True,          # Photometric augmentations for webcam lighting
    name='examguard_yolo11',
    save=True
)

print("[✓] Training complete! Best weights saved at: runs/detect/examguard_yolo11/weights/best.pt")

## Step 4: Convert and Export Directly to TensorFlow.js (`model.json`)

In [ ]:
from ultralytics import YOLO
import shutil

# 1. Load best trained model
best_model = YOLO('runs/detect/examguard_yolo11/weights/best.pt')

# 2. Export to TensorFlow.js format
export_path = best_model.export(format='tfjs')
print(f"[✓] Exported to TF.js format at: {export_path}")

# 3. Zip the model files for quick 1-click download
shutil.make_archive("/content/examguard_web_model", 'zip', export_path)
print("[✓] Created zip package: /content/examguard_web_model.zip")

## Step 5: Download the Web Model to Your Computer

In [ ]:
from google.colab import files

# Automatically downloads the zip to your Downloads folder!
# Extract it into your project: static/models/examguard_vision/
files.download('/content/examguard_web_model.zip')